# 01. 検出モデルの学習と評価(Colab)

`scripts/prepare_dataset.py split` で作ったデータセット(`data/dataset/yolo/`)を zip にして Google Drive に置き、
このノートブックで YOLOv8 をファインチューニングする。

1. ランタイム > ランタイムのタイプを変更 > T4 GPU
2. 下の設定セルの `DRIVE_DIR` を、zip を置いた Drive のフォルダに合わせる
3. 上から順に実行する。学習結果(重み・学習曲線・評価結果)は Drive の `runs/` に保存される

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/dataset/wbt'   # yolo.zip を置いたフォルダ
MODEL = 'yolov8s.pt'                        # n / s / m を比較する場合はここを変えて再実行
IMGSZ = 1280                                # ボールが小さいため 1280 以上にする
EPOCHS = 100
RUN_NAME = f'{MODEL.split(".")[0]}_{IMGSZ}'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics
!git clone -q https://github.com/ashina1992/wheelchair-basketball-tracking.git /content/wbt || true
!pip install -q -e /content/wbt
!unzip -q -o {DRIVE_DIR}/yolo.zip -d /content/dataset
!sed -i 's#^path: .*#path: /content/dataset#' /content/dataset/data.yaml
!cat /content/dataset/data.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data='/content/dataset/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=8, device=0,
    project=f'{DRIVE_DIR}/runs', name=RUN_NAME, exist_ok=True,
    patience=30, cos_lr=True,
    # 放送映像は左右反転しても成り立つが、上下反転は成り立たない
    fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=10,
)

## test セットでの評価

既製の COCO モデル(person→player、sports ball→ball に読み替え)と、学習したモデルを同じ方法で評価する。
結果の JSON は `docs/03_finetune.md` にまとめる。

In [ ]:
best = f'{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt'
!cd /content/wbt && python scripts/eval_detector.py /content/dataset --split test --imgsz {IMGSZ} \
    --weights {MODEL} {best} --out {DRIVE_DIR}/runs/{RUN_NAME}/eval_test.json

In [ ]:
# ultralytics 標準の評価(クラス別 mAP、混同行列、PR 曲線の画像が保存される)
metrics = YOLO(best).val(data='/content/dataset/data.yaml', split='test', imgsz=IMGSZ,
                         project=f'{DRIVE_DIR}/runs', name=f'{RUN_NAME}_test', exist_ok=True)
print(metrics.results_dict)